# Notebook 1: Data Collection

In this notebook we connect to the Open Asset Pricing database and download 
quintile portfolio returns for two value signals:

- **BM (Book-to-Market):** Compares a company's book value to its stock price
- **EP (Earnings-to-Price):** Compares a company's earnings to its stock price

We also download Fama-French factor data, which is the academic industry standard 
for measuring the value premium. All data is saved to CSV files so we don't have 
to re-download it in future notebooks.

In [12]:
pip install openassetpricing pandas-datareader fredapi


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import openassetpricing as openap
import pandas_datareader.data as web
from fredapi import Fred

### Step 1: Connect to Open Asset Pricing and Download BM Data
Book-to-Market (BM) sorts stocks into 5 groups: Q1 is growth (expensive), Q5 is value (cheap).

In [14]:
ap = openap.OpenAP()

bm = ap.dl_port('quintiles_ew', 'pandas', ['BM'])
print("BM data shape:", bm.shape)
print("Columns:", bm.columns.tolist())
print(bm.head())


Data is downloaded: 3s
BM data shape: (5292, 7)
Columns: ['signalname', 'port', 'date', 'ret', 'signallag', 'Nlong', 'Nshort']
  signalname port       date       ret  signallag  Nlong  Nshort
0         BM   01 1951-07-31  7.976745  -0.854527     65       0
1         BM   01 1951-08-31  3.926174  -0.854527     65       0
2         BM   01 1951-09-28  1.263965  -0.854527     65       0
3         BM   01 1951-10-31 -4.242974  -0.854527     65       0
4         BM   01 1951-11-30  1.147037  -0.854527     65       0


### Step 1 (continued): Download EP Data
Earnings-to-Price (EP) is a second way to define value: how much a company earns per dollar of stock price.

In [15]:
ep  = ap.dl_port('quintiles_ew', 'pandas', ['EP'])
cfp = ap.dl_port('quintiles_ew', 'pandas', ['CFP'])

print("EP shape:", ep.shape)
print("CFP shape:", cfp.shape)


Data is downloaded: 4s
One or more input predictors are not available.

Data is downloaded: 5s
EP shape: (5328, 7)
CFP shape: (0, 7)


### Step 2: Inspect the Data Structure
We check what the raw data looks like before reshaping it.

In [16]:
print(bm['port'].unique())
print(bm['signalname'].unique())

['01' '02' '03' '04' '05' 'LS']
['BM']


### Step 3: Reshape Data into Wide Format
We reorganize the data so each quintile (Q1–Q5) becomes its own column.
The 'spread' column = Q5 minus Q1, showing how much value beat growth each month.

In [17]:
def pivot_quintiles(df):
    df = df[df['port'].isin(['01','02','03','04','05'])].copy()
    wide = df.pivot_table(index='date', columns='port', values='ret')
    wide.columns = ['Q1','Q2','Q3','Q4','Q5']
    wide.index = pd.to_datetime(wide.index)
    wide['spread'] = wide['Q5'] - wide['Q1']
    return wide

bm_wide = pivot_quintiles(bm)
ep_wide = pivot_quintiles(ep)

print("BM reshaped:")
print(bm_wide.head())
print("\nEP reshaped:")
print(ep_wide.head())

BM reshaped:
                  Q1        Q2        Q3        Q4        Q5    spread
date                                                                  
1951-07-31  7.976745  5.931340  6.213661  8.151611  6.287282 -1.689463
1951-08-31  3.926174  5.614672  5.038845  5.396380  5.314608  1.388434
1951-09-28  1.263965  1.252177  1.710839  2.137429  2.608031  1.344066
1951-10-31 -4.242974 -1.622966 -1.832547 -1.950945 -2.712077  1.530897
1951-11-30  1.147037  2.506245  1.562823  0.526054  1.155120  0.008083

EP reshaped:
                  Q1        Q2        Q3         Q4        Q5    spread
date                                                                   
1951-01-31  6.167020  8.270067  6.522700  10.000200  3.221560 -2.945460
1951-02-28  0.338150  4.241250  1.660340  -0.797260  5.380617  5.042467
1951-03-31 -1.621843 -0.925857 -4.371183  -4.717586 -4.924314 -3.302471
1951-04-30  4.110740  2.827278  2.626644   2.178478  3.114070 -0.996670
1951-05-31 -2.940815  2.269158  2.254123  -1

### Step 4: Download Fama-French Factor Data
The Fama-French factors are the industry standard benchmark. HML (High Minus Low) 
is essentially the academic version of the value premium, we'll use it later to 
compare against our own BM and EP spreads.

In [18]:
import pandas_datareader.data as web

ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1963-01-01')[0]
ff3 = ff3 / 100  # convert from percent to decimal
ff3.index = pd.to_datetime(ff3.index.to_timestamp())

print(ff3.head())

            Mkt-RF     SMB     HML      RF
Date                                      
1963-01-01  0.0494  0.0301  0.0224  0.0025
1963-02-01 -0.0240  0.0046  0.0215  0.0023
1963-03-01  0.0308 -0.0258  0.0210  0.0023
1963-04-01  0.0451 -0.0126  0.0100  0.0025
1963-05-01  0.0176  0.0108  0.0255  0.0024


/tmp/ipykernel_13759/3485994894.py:3: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1963-01-01')[0]
/tmp/ipykernel_13759/3485994894.py:3: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1963-01-01')[0]


### Step 5: Save All Data to CSV

In [19]:
bm_wide.to_csv('bm_wide.csv')
ep_wide.to_csv('ep_wide.csv')
ff3.to_csv('ff3_factors.csv')